In [1]:
!pip install thop

In [2]:
import torch

num_classes = 15
IMAGE_SIZE = (224, 224)
# IMAGE_SIZE[0], IMAGE_SIZE[1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
from thop import profile
from thop import clever_format
import torch

def display_params_flops(model):
    #params
    num_params = sum(p.numel() for p in model.parameters())
    num_params_millions = num_params / 1e6
    print(f"Number of parameters in millions: {num_params_millions:.2f} M")

    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    num_params_millions = num_params / 1e6
    print(f"Number of trainable parameters in millions: {num_params_millions:.2f} M")


    #FLOPS
    input_size = (1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1])  


    # Move the model to GPU if available
    if torch.cuda.is_available():
        model = model.cuda()

    # Use thop.profile to count FLOPs
    input_tensor = torch.randn(*input_size)
    if torch.cuda.is_available():
        input_tensor = input_tensor.cuda()
    flops, params = profile(model, inputs=(input_tensor,))

    # Convert FLOPs to gigaFLOPs and format the results
    flops, params = clever_format([flops, params], "%.2f")
    print(f"FLOPs: {flops}, Params: {params}")
    

### CNN Models

### 1. VGG-19

In [4]:
import torch
import torch.nn as nn
import torchvision.models as models


class VGG19Model(nn.Module):
    def __init__(self, num_classes, fine_tune=False):
        super(VGG19Model, self).__init__()

        # Add a convolutional layer at the top
        self.conv = nn.Conv2d(1, 3, kernel_size=3, stride=1, padding=1)  # Assuming input is grayscale (1 channel)

        self.vgg19 = models.vgg19(pretrained=True)
        
        if not fine_tune:
            # Freeze all layers except classifier layers
            for param in self.vgg19.parameters():
                param.requires_grad = False

            # Unfreeze the classifier layers
            for param in self.vgg19.classifier.parameters():
                param.requires_grad = True
            

        # Get the number of input features for the final fully connected layer
        in_features = self.vgg19.classifier[6].in_features

        # Replace the final fully connected layer with a new one for the specified number of classes
        self.vgg19.classifier[6] = nn.Linear(in_features, num_classes)
        
        

    def forward(self, x):
        x = self.conv(x)
        x = self.vgg19(x)
        return x
    
    
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VGG19Model(num_classes, fine_tune=False) # change fine_tune as required
model.to(device)

# print(model)
print()

x = torch.randn(1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]).to(device)


output = model(x)
print("Model output's shape:", output.shape)
print(output) # logits 
display_params_flops(model)

/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:03<00:00, 148MB/s]



Model output's shape: torch.Size([1, 15])
tensor([[ 0.1141, -0.1989,  0.0149,  0.3392,  0.0469, -0.3226,  0.3937,  0.6723,
         -0.3124, -0.3450,  0.0357, -0.4151, -0.0462,  0.2496,  0.0574]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Number of parameters in millions: 139.63 M
Number of trainable parameters in millions: 119.61 M
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
FLOPs: 19.63G, Params: 139.63M


### 2. EfficientNet

In [5]:
!pip install efficientnet_pytorch # for installing efficientnet model

import torch
import torch.nn as nn
import torchvision.models as models
from efficientnet_pytorch import EfficientNet

class EfficientNetModel(nn.Module):
    def __init__(self, num_classes, fine_tune=False):
        super(EfficientNetModel, self).__init__()
        self.efficientnet = EfficientNet.from_pretrained('efficientnet-b0', num_classes=num_classes, in_channels=1)
        
        if not fine_tune:
            for param in self.efficientnet.parameters():
                param.requires_grad = False

            for param in self.efficientnet._fc.parameters():
                param.requires_grad = True

        
        
    def forward(self, x):
        x = self.efficientnet(x)
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EfficientNetModel(num_classes, fine_tune=False)
model.to(device)

# print(model)
print()

x = torch.randn(1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]).to(device)


output = model(x)
print("Model output's shape:", output.shape)
print(output) # logits 
display_params_flops(model)

  Preparing metadata (setup.py) ... - done
  Created wheel for efficientnet_pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16428 sha256=5aa126a5656fd926a8c8974a568d3b2086ddf3843a5f110abab1d0e0006f16c4
  Stored in directory: /root/.cache/pip/wheels/03/3f/e9/911b1bc46869644912bda90a56bcf7b960f20b5187feea3baf
Successfully built efficientnet_pytorch


Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth
100%|██████████| 20.4M/20.4M [00:00<00:00, 155MB/s]


Loaded pretrained weights for efficientnet-b0

Model output's shape: torch.Size([1, 15])
tensor([[-0.0184, -0.1523,  0.0733,  0.0891, -0.0507, -0.0303, -0.0032, -0.0084,
          0.0310, -0.0577, -0.0176,  0.0120, -0.0025, -0.0212, -0.0970]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Number of parameters in millions: 4.03 M
Number of trainable parameters in millions: 0.02 M
[INFO] Register zero_ops() for <class 'torch.nn.modules.padding.ZeroPad2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
FLOPs: 27.03M, Params: 61.23K


### 3. ResNet

In [6]:
import torch
import torch.nn as nn
import torchvision.models as models

class ResNetModel(nn.Module):
    def __init__(self, num_classes, fine_tune=False):
        super(ResNetModel, self).__init__()        
         # Add a convolutional layer at the top
        self.conv = nn.Conv2d(1, 3, kernel_size=3, stride=1, padding=1)  # Assuming input is grayscale (1 channel)

        self.resnet = models.resnet152(pretrained=True)
        
        if not fine_tune:
            for param in self.resnet.parameters():
                param.requires_grad = False
            
            for param in self.resnet.fc.parameters():
                param.requires_grad = True
        

        in_features = self.resnet.fc.in_features

        # Replace the final fully connected layer with a new one for the specified number of classes
        self.resnet.fc = nn.Linear(in_features, num_classes)
        
        
    def forward(self, x):
        x = self.conv(x)
        x = self.resnet(x)
        return x
    
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNetModel(num_classes, fine_tune=False)
model.to(device)

# print(model)
print()

x = torch.randn(1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]).to(device)


output = model(x)
print("Model output's shape:", output.shape)
print(output) # logits 
display_params_flops(model)

/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet152-394f9c45.pth" to /root/.cache/torch/hub/checkpoints/resnet152-394f9c45.pth
100%|██████████| 230M/230M [00:01<00:00, 136MB/s]



Model output's shape: torch.Size([1, 15])
tensor([[ 0.3527,  0.2662, -0.2867,  0.5065,  0.1777, -0.1360,  0.1384,  0.3014,
          0.1933,  0.0919, -0.0363,  0.1002,  0.3396,  0.4503, -0.2354]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Number of parameters in millions: 58.17 M
Number of trainable parameters in millions: 0.03 M
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
FLOPs: 11.60G, Params: 58.17M


### 4. CoatNet ( 512 x 512 image size not compatible )

In [7]:
import torch
import torch.nn as nn
import torchvision.models as models

from torch import nn
from timm import create_model


class CoatNetModel(nn.Module):
    def __init__(self, num_classes, fine_tune=False):
        super(CoatNetModel, self).__init__()
        self.coatnet = create_model(
            'timm/coatnet_3_rw_224.sw_in12k', # accepts only 224x224 images
            pretrained=True,
            num_classes=num_classes,
            in_chans=1
        )
        
        if not fine_tune:
            for param in self.coatnet.parameters():
                param.requires_grad = False
            
            for param in self.coatnet.head.parameters():
                param.requires_grad = True
        

    def forward(self, x):
        x = self.coatnet(x)
        return x
    
    
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CoatNetModel(num_classes, fine_tune=False)
model.to(device)

# print(model)
print()

x = torch.randn(1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]).to(device)


output = model(x)
print("Model output's shape:", output.shape)
print(output) # logits
display_params_flops(model)

model.safetensors:   0%|          | 0.00/727M [00:00<?, ?B/s]


Model output's shape: torch.Size([1, 15])
tensor([[ 1.0493,  0.4044,  0.6934, -2.3854,  2.4097,  0.5271, -2.4023, -2.9863,
         -0.0977,  1.1776, -2.9211, -1.2702, -1.6801,  2.0718, -0.2862]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Number of parameters in millions: 163.66 M
Number of trainable parameters in millions: 0.02 M
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_avgpool() for <class 'torch.nn.modules.pooling.AvgPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
FLOPs: 32.49G, Params: 163.28M


### 5. ConvNeXt

In [8]:
import torch
import torch.nn as nn
import torchvision.models as models

from torch import nn
from timm import create_model

class ConvNeXtModel(nn.Module):
    def __init__(self, num_classes, model_name="convnext_tiny", pretrained=True, fine_tune=False):
        super(ConvNeXtModel, self).__init__()
        self.convnext_model = create_model(
            model_name, 
            pretrained=pretrained,
            num_classes=num_classes,
            in_chans=1
        )
        
        if not fine_tune:
            for param in self.convnext_model.parameters():
                param.requires_grad = False
            
            for param in self.convnext_model.head.parameters():
                param.requires_grad = True
        
        
    def forward(self, x):
        x = self.convnext_model(x)
        return x
    
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ConvNeXtModel(num_classes, fine_tune=False)
model.to(device)

# print(model)
print()

x = torch.randn(1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]).to(device)


output = model(x)
print("Model output's shape:", output.shape)
print(output) # logits 
display_params_flops(model)

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]


Model output's shape: torch.Size([1, 15])
tensor([[-0.9079,  0.9408, -0.0791,  0.0431, -0.3826, -0.4962,  0.5288,  0.7123,
         -1.1503,  0.4666, -0.7218,  0.7928,  0.0454,  0.4059,  1.0176]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Number of parameters in millions: 27.83 M
Number of trainable parameters in millions: 0.01 M
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
FLOPs: 4.45G, Params: 27.81M


### 6. DenseNet

In [9]:
import torch
import torch.nn as nn
import torchvision.models as models

from torch import nn
from timm import create_model

class DenseNetModel(nn.Module):
    def __init__(self, num_classes, pretrained=True, fine_tune=False):
        super(DenseNetModel, self).__init__()
        self.densenet = create_model(
            'densenet121.tv_in1k', 
            pretrained=pretrained,
            num_classes=num_classes,
            in_chans=1
        )

        if not fine_tune:
            for param in self.densenet.parameters():
                param.requires_grad = False
            
            for param in self.densenet.global_pool.parameters():
                param.requires_grad = True
                
            for param in self.densenet.head_drop.parameters():
                param.requires_grad = True
                
            for param in self.densenet.classifier.parameters():
                param.requires_grad = True
        
        
    def forward(self, x):
        x = self.densenet(x)
        return x
    
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DenseNetModel(num_classes, fine_tune=False)
model.to(device)

# print(model)
print()

x = torch.randn(1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]).to(device)


output = model(x)
print("Model output's shape:", output.shape)
print(output) # logits 
display_params_flops(model)

model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]


Model output's shape: torch.Size([1, 15])
tensor([[ 0.2303,  0.3872, -0.1187, -0.0159,  0.2146,  0.4215, -0.4641, -0.2141,
          0.3579, -0.0363, -0.2009,  0.7574, -0.1098, -0.0292, -0.1578]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Number of parameters in millions: 6.96 M
Number of trainable parameters in millions: 0.02 M
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register count_avgpool() for <class 'torch.nn.modules.pooling.AvgPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
FLOPs: 2.75G, Params: 6.

### Transformers

### 1. Swin Transformer (512 x 512 not compatible) only 224 or 384

In [10]:
import torch
import torch.nn as nn
import torchvision.models as models

from torch import nn
from timm import create_model


class SwinTransformerModel(nn.Module):
    def __init__(self, num_classes, fine_tune=False):
        super(SwinTransformerModel, self).__init__()
        self.swin = create_model(
            'swin_large_patch4_window7_224.ms_in22k', 
            pretrained=True,
            num_classes=num_classes,
            in_chans=1
        )
        
        if not fine_tune:
            for param in self.swin.parameters():
                param.requires_grad = False
            
            for param in self.swin.head.parameters():
                param.requires_grad = True
        

    def forward(self, x):
        x = self.swin(x)
        return x
    
    
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SwinTransformerModel(num_classes, fine_tune=False)
model.to(device)

# print(model)
print()

x = torch.randn(1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]).to(device)


output = model(x)
print("Model output's shape:", output.shape)
print(output) # logits 
display_params_flops(model)

model.safetensors:   0%|          | 0.00/916M [00:00<?, ?B/s]


Model output's shape: torch.Size([1, 15])
tensor([[-0.2104,  0.0284, -0.3546, -0.2533,  0.2120,  0.7453, -0.1186, -0.7598,
          0.3066,  0.2589, -0.0577,  0.7392, -0.3076, -0.0241, -0.2682]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Number of parameters in millions: 195.01 M
Number of trainable parameters in millions: 0.02 M
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_softmax() for <class 'torch.nn.modules.activation.Softmax'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
FLOPs: 34.06G, Params: 194.92M


### 2. MViT (512 x 512 image not supported) 

In [11]:
import timm
import torch.nn as nn

class MViT(nn.Module):
    def __init__(self, num_classes, pretrained=True, fine_tune=False):
        super(MViT, self).__init__()
        self.mvit = timm.create_model('mvitv2_base.fb_in1k', 
           pretrained=pretrained, 
           num_classes=num_classes,
            in_chans=1)
        
        
        if not fine_tune:
            for param in self.mvit.parameters():
                param.requires_grad = False
            
            for param in self.mvit.head.parameters():
                param.requires_grad = True
        

    def forward(self, x):
        return self.mvit(x)
    

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MViT(num_classes, fine_tune=False)
model.to(device)

# print(model)
print()

x = torch.randn(1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]).to(device)


output = model(x)
print("Model output's shape:", output.shape)
print(output) # logits    
display_params_flops(model)

model.safetensors:   0%|          | 0.00/206M [00:00<?, ?B/s]


Model output's shape: torch.Size([1, 15])
tensor([[ 0.0954,  0.0023, -0.1967, -0.0303, -0.0759, -0.0975,  0.2859, -0.1323,
          0.3687,  0.3437, -0.0465,  0.2573, -0.2109, -0.0827,  0.0485]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Number of parameters in millions: 50.71 M
Number of trainable parameters in millions: 0.01 M
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
FLOPs: 8.83G, Params: 50.48M


### 3. DaViT

In [12]:
import torch
import torch.nn as nn
import torchvision.models as models

from torch import nn
from timm import create_model


class DaViT(nn.Module):
    def __init__(self, num_classes, fine_tune=False):
        super(DaViT, self).__init__()
        self.davit = create_model(
            'davit_base.msft_in1k', 
            pretrained=True,
            num_classes=num_classes,
            in_chans=1
        )
        

        if not fine_tune:
            for param in self.davit.parameters():
                param.requires_grad = False
            
            for param in self.davit.head.parameters():
                param.requires_grad = True
        
        

    def forward(self, x):
        x = self.davit(x)
        return x
    
    
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DaViT(num_classes, fine_tune=False)
model.to(device)

# print(model)
print()

x = torch.randn(1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]).to(device)


output = model(x)
print("Model output's shape:", output.shape)
print(output) # logits 
display_params_flops(model)

pytorch_model.bin:   0%|          | 0.00/352M [00:00<?, ?B/s]


Model output's shape: torch.Size([1, 15])
tensor([[-0.2269,  0.1547, -0.0736,  0.1810,  0.1348, -0.2158, -0.2421,  0.1566,
          0.0277,  0.1135,  0.0463, -0.1975, -0.2434,  0.2884,  0.2682]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Number of parameters in millions: 86.93 M
Number of trainable parameters in millions: 0.02 M
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_softmax() for <class 'torch.nn.modules.activation.Softmax'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
FLOPs: 15.18G, Params: 86.88M


### 4. PVT

In [13]:
import timm
import torch.nn as nn


class PVT(nn.Module):
    def __init__(self, num_classes, pretrained=True, fine_tune=False):
        super(PVT, self).__init__()
        self.pvt = timm.create_model('pvt_v2_b5', 
           pretrained=pretrained, 
           num_classes=num_classes,
           in_chans=1)
        
#         model.pvt.head_drop
#         model.pvt.head

        if not fine_tune:
            for param in self.pvt.parameters():
                param.requires_grad = False
            
            for param in self.pvt.head_drop.parameters():
                param.requires_grad = True
            
            for param in self.pvt.head.parameters():
                param.requires_grad = True
        

    def forward(self, x):
        return self.pvt(x)

    
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PVT(num_classes, fine_tune=False)
model.to(device)

# print(model)
print()

x = torch.randn(1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]).to(device)


output = model(x)
print("Model output's shape:", output.shape)
print(output) # logits 
display_params_flops(model)

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]


Model output's shape: torch.Size([1, 15])
tensor([[-0.8003,  0.2739, -0.2441,  0.9140,  0.2193,  0.1956, -0.4293, -0.3385,
         -0.1421, -0.2328, -0.8976,  0.5361, -0.1937,  0.5554, -0.2802]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Number of parameters in millions: 81.44 M
Number of trainable parameters in millions: 0.01 M
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
FLOPs: 11.33G, Params: 81.38M


### 5. GC ViT (Not pretrained) 512 compatible version is very time consuming [1 hidden cell]

In [14]:
#!/usr/bin/env python3

# Copyright (c) 2023, NVIDIA CORPORATION.  All rights reserved.
#
# NVIDIA CORPORATION and its licensors retain all intellectual property
# and proprietary rights in and to this software, related documentation
# and any modifications thereto.  Any use, reproduction, disclosure or
# distribution of this software and related documentation without an express
# license agreement from NVIDIA CORPORATION is strictly prohibited.
# written by Ali Hatamizadeh and Pavlo Molchanov from NVResearch


import torch
import torch.nn as nn
from timm.models.layers import trunc_normal_, DropPath, to_2tuple
from timm.models._registry import register_model
from timm.models._builder import build_model_with_cfg


def _cfg(url='', **kwargs):
    return {'url': url,
            'num_classes': num_classes, # set num_classes here
            'input_size': (1, IMAGE_SIZE[0], IMAGE_SIZE[1]),  # adjust input image size here
            'pool_size': None,
            'crop_pct': 0.875, 
            'interpolation': 'bicubic', 
            'fixed_input_size': True,
            'mean': (0.485, 0.456, 0.406), 
            'std': (0.229, 0.224, 0.225),
            **kwargs
            }


default_cfgs = {
    'gc_vit_xxtiny': _cfg(url='https://huggingface.co/nvidia/GCViT/resolve/main/gcvit_1k_xxtiny.pth.tar',
                          crop_pct=1.0, 
                          input_size=(3, 224, 224), 
                          crop_mode= 'center'),
    'gc_vit_xtiny': _cfg(url='https://huggingface.co/nvidia/GCViT/resolve/main/gcvit_1k_xtiny.pth.tar',
                         crop_pct=0.875, 
                         input_size=(3, 224, 224), 
                         crop_mode= 'center'),
    'gc_vit_tiny': _cfg(url='https://huggingface.co/nvidia/GCViT/resolve/main/gcvit_1k_tiny.pth.tar',
                        crop_pct=1.0, 
                        input_size=(3, 224, 224), 
                        crop_mode= 'center'),
    'gc_vit_tiny2': _cfg(url='https://huggingface.co/nvidia/GCViT/resolve/main/gcvit_1k_tiny2.pth.tar',
                         crop_pct=0.875, 
                         input_size=(3, 224, 224), 
                         crop_mode= 'center'),
    'gc_vit_small': _cfg(url='https://huggingface.co/nvidia/GCViT/resolve/main/gcvit_1k_small.pth.tar',
                         crop_pct=0.875, 
                         input_size=(3, 224, 224), 
                         crop_mode= 'center'),
    'gc_vit_small2': _cfg(url='https://huggingface.co/nvidia/GCViT/resolve/main/gcvit_1k_small2.pth.tar',
                          crop_pct=0.875, 
                          input_size=(3, 224, 224), 
                          crop_mode= 'center'),
    'gc_vit_base': _cfg(url='https://huggingface.co/nvidia/GCViT/resolve/main/gcvit_1k_base.pth.tar',
                        crop_pct=1.0, 
                        input_size=(3, 224, 224), 
                        crop_mode= 'center'),
    'gc_vit_large': _cfg(url='https://huggingface.co/nvidia/GCViT/resolve/main/gcvit_1k_large.pth.tar',
                         crop_pct=1.0, 
                         input_size=(3, 224, 224), 
                         crop_mode= 'center'),
    'gc_vit_large_224_21k': _cfg(url='https://drive.google.com/uc?export=download&id=1maGDr6mJkLyRTUkspMzCgSlhDzNRFGEf', 
                                 crop_pct=1.0, 
                                 input_size=(3, 224, 224), 
                                 crop_mode= 'center'),
    'gc_vit_large_384_21k': _cfg(url='https://huggingface.co/nvidia/GCViT/resolve/main/gcvit_21k_large_384.pth.tar', 
                                 crop_pct=1.0, 
                                 input_size=(3, 384, 384), 
                                 crop_mode='squash'),
    'gc_vit_large_512_21k': _cfg(url='https://huggingface.co/nvidia/GCViT/resolve/main/gcvit_21k_large_512.pth.tar', 
                                 crop_pct=1.0, 
                                 input_size=(3, 512, 512), 
                                 crop_mode='squash'),                             
}


def _to_channel_last(x):
    """
    Args:
        x: (B, C, H, W)

    Returns:
        x: (B, H, W, C)
    """
    return x.permute(0, 2, 3, 1)


def _to_channel_first(x):
    """
    Args:
        x: (B, H, W, C)

    Returns:
        x: (B, C, H, W)
    """
    return x.permute(0, 3, 1, 2)


def window_partition(x, window_size, h_w, w_w):
    """
    Args:
        x: (B, H, W, C)
        window_size: window size

    Returns:
        local window features (num_windows*B, window_size, window_size, C)
    """
    B, H, W, C = x.shape
    x = x.view(B, h_w, window_size, w_w, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W, h_w, w_w, B):
    """
    Args:
        windows: local window features (num_windows*B, window_size, window_size, C)
        window_size: Window size
        H: Height of image
        W: Width of image

    Returns:
        x: (B, H, W, C)
    """
    # B = int(windows.shape[0] // (H * W // window_size // window_size))
    x = windows.view(B, h_w, w_w, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


class Mlp(nn.Module):
    """
    Multi-Layer Perceptron (MLP) block
    """

    def __init__(self,
                 in_features,
                 hidden_features=None,
                 out_features=None,
                 act_layer=nn.GELU,
                 drop=0.):
        """
        Args:
            in_features: input features dimension.
            hidden_features: hidden features dimension.
            out_features: output features dimension.
            act_layer: activation function.
            drop: dropout rate.
        """

        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class SE(nn.Module):
    """
    Squeeze and excitation block
    """

    def __init__(self,
                 inp,
                 oup,
                 expansion=0.25):
        """
        Args:
            inp: input features dimension.
            oup: output features dimension.
            expansion: expansion ratio.
        """

        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(oup, int(inp * expansion), bias=False),
            nn.GELU(),
            nn.Linear(int(inp * expansion), oup, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class ReduceSize(nn.Module):
    """
    Down-sampling block based on: "Hatamizadeh et al.,
    Global Context Vision Transformers <https://arxiv.org/abs/2206.09959>"
    """

    def __init__(self,
                 dim,
                 norm_layer=nn.LayerNorm,
                 keep_dim=False):
        """
        Args:
            dim: feature size dimension.
            norm_layer: normalization layer.
            keep_dim: bool argument for maintaining the resolution.
        """

        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1,
                      groups=dim, bias=False),
            nn.GELU(),
            SE(dim, dim),
            nn.Conv2d(dim, dim, 1, 1, 0, bias=False),
        )
        if keep_dim:
            dim_out = dim
        else:
            dim_out = 2 * dim
        self.reduction = nn.Conv2d(dim, dim_out, 3, 2, 1, bias=False)
        self.norm2 = norm_layer(dim_out)
        self.norm1 = norm_layer(dim)

    def forward(self, x):
        x = x.contiguous()
        x = self.norm1(x)
        x = _to_channel_first(x)
        x = x + self.conv(x)
        x = self.reduction(x)
        x = _to_channel_last(x)
        x = self.norm2(x)
        return x


class PatchEmbed(nn.Module):
    """
    Patch embedding block based on: "Hatamizadeh et al.,
    Global Context Vision Transformers <https://arxiv.org/abs/2206.09959>"
    """

    def __init__(self, in_chans=3, dim=96):
        """
        Args:
            in_chans: number of input channels.
            dim: feature size dimension.
        """

        super().__init__()
        self.proj = nn.Conv2d(in_chans, dim, 3, 2, 1)
        self.conv_down = ReduceSize(dim=dim, keep_dim=True)

    def forward(self, x):
        x = self.proj(x)
        x = _to_channel_last(x)
        x = self.conv_down(x)
        return x


class FeatExtract(nn.Module):
    """
    Feature extraction block based on: "Hatamizadeh et al.,
    Global Context Vision Transformers <https://arxiv.org/abs/2206.09959>"
    """

    def __init__(self, dim, keep_dim=False):
        """
        Args:
            dim: feature size dimension.
            keep_dim: bool argument for maintaining the resolution.
        """

        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1,
                      groups=dim, bias=False),
            nn.GELU(),
            SE(dim, dim),
            nn.Conv2d(dim, dim, 1, 1, 0, bias=False),
        )
        if not keep_dim:
            self.pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.keep_dim = keep_dim

    def forward(self, x):
        x = x.contiguous()
        x = x + self.conv(x)
        if not self.keep_dim:
            x = self.pool(x)
        return x


class WindowAttention(nn.Module):
    """
    Local window attention based on: "Liu et al.,
    Swin Transformer: Hierarchical Vision Transformer using Shifted Windows
    <https://arxiv.org/abs/2103.14030>"
    """

    def __init__(self,
                 dim,
                 num_heads,
                 window_size,
                 qkv_bias=True,
                 qk_scale=None,
                 attn_drop=0.,
                 proj_drop=0.,
                 ):
        """
        Args:
            dim: feature size dimension.
            num_heads: number of attention head.
            window_size: window size.
            qkv_bias: bool argument for query, key, value learnable bias.
            qk_scale: bool argument to scaling query, key.
            attn_drop: attention dropout rate.
            proj_drop: output dropout rate.
        """

        super().__init__()
        window_size = (window_size, window_size)
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = torch.div(dim, num_heads, rounding_mode='floor')
        self.scale = qk_scale or head_dim ** -0.5
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))
        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w]))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += self.window_size[0] - 1
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer("relative_position_index", relative_position_index)
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        trunc_normal_(self.relative_position_bias_table, std=.02)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, q_global):
        B_, N, C = x.shape
        head_dim = torch.div(C, self.num_heads, rounding_mode='floor')
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))
        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1)
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class WindowAttentionGlobal(nn.Module):
    """
    Global window attention based on: "Hatamizadeh et al.,
    Global Context Vision Transformers <https://arxiv.org/abs/2206.09959>"
    """

    def __init__(self,
                 dim,
                 num_heads,
                 window_size,
                 qkv_bias=True,
                 qk_scale=None,
                 attn_drop=0.,
                 proj_drop=0.,
                 ):
        """
        Args:
            dim: feature size dimension.
            num_heads: number of attention head.
            window_size: window size.
            qkv_bias: bool argument for query, key, value learnable bias.
            qk_scale: bool argument to scaling query, key.
            attn_drop: attention dropout rate.
            proj_drop: output dropout rate.
        """

        super().__init__()
        window_size = (window_size, window_size)
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = torch.div(dim, num_heads, rounding_mode='floor')
        self.scale = qk_scale or head_dim ** -0.5
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))
        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w]))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += self.window_size[0] - 1
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer("relative_position_index", relative_position_index)
        self.qkv = nn.Linear(dim, dim * 2, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        trunc_normal_(self.relative_position_bias_table, std=.02)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, q_global):
        B_, N, C = x.shape
        B = q_global.shape[0]
        head_dim = torch.div(C, self.num_heads, rounding_mode='floor')
        B_dim = torch.div(B_, B, rounding_mode='floor')
        kv = self.qkv(x).reshape(B_, N, 2, self.num_heads, head_dim).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        q_global = q_global.repeat(1, B_dim, 1, 1, 1)
        q = q_global.reshape(B_, self.num_heads, N, head_dim)
        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))
        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1)
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class GCViTBlock(nn.Module):
    """
    GCViT block based on: "Hatamizadeh et al.,
    Global Context Vision Transformers <https://arxiv.org/abs/2206.09959>"
    """

    def __init__(self,
                 dim,
                 input_resolution,
                 num_heads,
                 window_size=7,
                 mlp_ratio=4.,
                 qkv_bias=True,
                 qk_scale=None,
                 drop=0.,
                 attn_drop=0.,
                 drop_path=0.,
                 act_layer=nn.GELU,
                 attention=WindowAttentionGlobal,
                 norm_layer=nn.LayerNorm,
                 layer_scale=None,
                 ):
        """
        Args:
            dim: feature size dimension.
            input_resolution: input image resolution.
            num_heads: number of attention head.
            window_size: window size.
            mlp_ratio: MLP ratio.
            qkv_bias: bool argument for query, key, value learnable bias.
            qk_scale: bool argument to scaling query, key.
            drop: dropout rate.
            attn_drop: attention dropout rate.
            drop_path: drop path rate.
            act_layer: activation function.
            attention: attention block type.
            norm_layer: normalization layer.
            layer_scale: layer scaling coefficient.
        """

        super().__init__()
        self.window_size = window_size
        self.norm1 = norm_layer(dim)

        self.attn = attention(dim,
                              num_heads=num_heads,
                              window_size=window_size,
                              qkv_bias=qkv_bias,
                              qk_scale=qk_scale,
                              attn_drop=attn_drop,
                              proj_drop=drop,
                              )

        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio), act_layer=act_layer, drop=drop)
        self.layer_scale = False
        if layer_scale is not None and type(layer_scale) in [int, float]:
            self.layer_scale = True
            self.gamma1 = nn.Parameter(layer_scale * torch.ones(dim), requires_grad=True)
            self.gamma2 = nn.Parameter(layer_scale * torch.ones(dim), requires_grad=True)
        else:
            self.gamma1 = 1.0
            self.gamma2 = 1.0

        inp_w = torch.div(input_resolution, window_size, rounding_mode='floor')
        self.num_windows = int(inp_w * inp_w)

    def forward(self, x, q_global):
        B, H, W, C = x.shape
        shortcut = x
        x = self.norm1(x)
        h_w = torch.div(H, self.window_size, rounding_mode='floor')
        w_w = torch.div(W, self.window_size, rounding_mode='floor')
        x_windows = window_partition(x, self.window_size, h_w, w_w)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        attn_windows = self.attn(x_windows, q_global)
        x = window_reverse(attn_windows, self.window_size, H, W, h_w, w_w, B)
        x = shortcut + self.drop_path(self.gamma1 * x)
        x = x + self.drop_path(self.gamma2 * self.mlp(self.norm2(x)))
        return x


class GlobalQueryGen(nn.Module):
    """
    Global query generator based on: "Hatamizadeh et al.,
    Global Context Vision Transformers <https://arxiv.org/abs/2206.09959>"
    """

    def __init__(self,
                 dim,
                 input_resolution,
                 image_resolution,
                 window_size,
                 num_heads):
        """
        Args:
            dim: feature size dimension.
            input_resolution: input image resolution.
            window_size: window size.
            num_heads: number of heads.

        For instance, repeating log(56/7) = 3 blocks, with input window dimension 56 and output window dimension 7 at
        down-sampling ratio 2. Please check Fig.5 of GC ViT paper for details.
        """

        super().__init__()
        if input_resolution == image_resolution//4:
            self.to_q_global = nn.Sequential(
                FeatExtract(dim, keep_dim=False),
                FeatExtract(dim, keep_dim=False),
                FeatExtract(dim, keep_dim=False),
            )

        elif input_resolution == image_resolution//8:
            self.to_q_global = nn.Sequential(
                FeatExtract(dim, keep_dim=False),
                FeatExtract(dim, keep_dim=False),
            )

        elif input_resolution == image_resolution//16:

            if window_size == input_resolution:
                self.to_q_global = nn.Sequential(
                    FeatExtract(dim, keep_dim=True)
                )

            else:
                self.to_q_global = nn.Sequential(
                    FeatExtract(dim, keep_dim=True)
                )

        elif input_resolution == image_resolution//32:
            self.to_q_global = nn.Sequential(
                FeatExtract(dim, keep_dim=True)
            )

        self.resolution = input_resolution
        self.num_heads = num_heads
        self.N = window_size * window_size
        self.dim_head = torch.div(dim, self.num_heads, rounding_mode='floor')

    def forward(self, x):
        x = _to_channel_last(self.to_q_global(x))
        B = x.shape[0]
        x = x.reshape(B, 1, self.N, self.num_heads, self.dim_head).permute(0, 1, 3, 2, 4)
        return x


class GCViTLayer(nn.Module):
    """
    GCViT layer based on: "Hatamizadeh et al.,
    Global Context Vision Transformers <https://arxiv.org/abs/2206.09959>"
    """

    def __init__(self,
                 dim,
                 depth,
                 input_resolution,
                 image_resolution,
                 num_heads,
                 window_size,
                 downsample=True,
                 mlp_ratio=4.,
                 qkv_bias=True,
                 qk_scale=None,
                 drop=0.,
                 attn_drop=0.,
                 drop_path=0.,
                 norm_layer=nn.LayerNorm,
                 layer_scale=None):
        """
        Args:
            dim: feature size dimension.
            depth: number of layers in each stage.
            input_resolution: input image resolution.
            window_size: window size in each stage.
            downsample: bool argument for down-sampling.
            mlp_ratio: MLP ratio.
            num_heads: number of heads in each stage.
            qkv_bias: bool argument for query, key, value learnable bias.
            qk_scale: bool argument to scaling query, key.
            drop: dropout rate.
            attn_drop: attention dropout rate.
            drop_path: drop path rate.
            norm_layer: normalization layer.
            layer_scale: layer scaling coefficient.
        """

        super().__init__()
        self.blocks = nn.ModuleList([
            GCViTBlock(dim=dim,
                       num_heads=num_heads,
                       window_size=window_size,
                       mlp_ratio=mlp_ratio,
                       qkv_bias=qkv_bias,
                       qk_scale=qk_scale,
                       attention=WindowAttention if (i % 2 == 0) else WindowAttentionGlobal,
                       drop=drop,
                       attn_drop=attn_drop,
                       drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                       norm_layer=norm_layer,
                       layer_scale=layer_scale,
                       input_resolution=input_resolution)
            for i in range(depth)])
        self.downsample = None if not downsample else ReduceSize(dim=dim, norm_layer=norm_layer)
        self.q_global_gen = GlobalQueryGen(dim, input_resolution, image_resolution, window_size, num_heads)

    def forward(self, x):
        q_global = self.q_global_gen(_to_channel_first(x))
        for blk in self.blocks:
            x = blk(x, q_global)
        if self.downsample is None:
            return x
        return self.downsample(x)


class GCViT(nn.Module):
    """
    GCViT based on: "Hatamizadeh et al.,
    Global Context Vision Transformers <https://arxiv.org/abs/2206.09959>"
    """

    def __init__(self,
                 dim,
                 depths,
                 window_size,
                 mlp_ratio,
                 num_heads,
                 resolution=224,
                 drop_path_rate=0.2,
                 in_chans=3,
                 num_classes=1000,
                 qkv_bias=True,
                 qk_scale=None,
                 drop_rate=0.,
                 attn_drop_rate=0.,
                 norm_layer=nn.LayerNorm,
                 layer_scale=None,
                 **kwargs):
        """
        Args:
            dim: feature size dimension.
            depths: number of layers in each stage.
            window_size: window size in each stage.
            mlp_ratio: MLP ratio.
            num_heads: number of heads in each stage.
            resolution: input image resolution.
            drop_path_rate: drop path rate.
            in_chans: number of input channels.
            num_classes: number of classes.
            qkv_bias: bool argument for query, key, value learnable bias.
            qk_scale: bool argument to scaling query, key.
            drop_rate: dropout rate.
            attn_drop_rate: attention dropout rate.
            norm_layer: normalization layer.
            layer_scale: layer scaling coefficient.
        """
        super().__init__()

        num_features = int(dim * 2 ** (len(depths) - 1))
        self.num_classes = num_classes
        self.patch_embed = PatchEmbed(in_chans=in_chans, dim=dim)
        self.pos_drop = nn.Dropout(p=drop_rate)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        self.levels = nn.ModuleList()
        for i in range(len(depths)):
            level = GCViTLayer(dim=int(dim * 2 ** i),
                               depth=depths[i],
                               num_heads=num_heads[i],
                               window_size=window_size[i],
                               mlp_ratio=mlp_ratio,
                               qkv_bias=qkv_bias,
                               qk_scale=qk_scale,
                               drop=drop_rate, attn_drop=attn_drop_rate,
                               drop_path=dpr[sum(depths[:i]):sum(depths[:i + 1])],
                               norm_layer=norm_layer,
                               downsample=(i < len(depths) - 1),
                               layer_scale=layer_scale,
                               input_resolution=int(2 ** (-2 - i) * resolution),
                               image_resolution=resolution)
            self.levels.append(level)
        self.norm = norm_layer(num_features)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(num_features, num_classes) if num_classes > 0 else nn.Identity()
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    @torch.jit.ignore
    def no_weight_decay_keywords(self):
        return {'rpb'}

    def forward_features(self, x):
        x = self.patch_embed(x)
        x = self.pos_drop(x)

        for level in self.levels:
            x = level(x)

        x = self.norm(x)
        x = _to_channel_first(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return x

    def forward(self, x):
        x = self.forward_features(x)
        x = self.head(x)
        return x



def _create_gc_vit(variant, pretrained=False, **kwargs):

    return build_model_with_cfg(
        GCViT,
        variant,
        pretrained,
        **kwargs,
    )


@register_model
def gc_vit_xxtiny(pretrained=False, **kwargs) -> GCViT:
    drop_path_rate = kwargs.pop("drop_path_rate", 0.2)
    model_kwargs = dict(depths=[2, 2, 6, 2],
                        num_heads=[2, 4, 8, 16],
                        window_size=[7, 7, 14, 7],
                        dim=64,
                        mlp_ratio=3,
                        drop_path_rate=drop_path_rate,
                        **kwargs
                        )
    model = _create_gc_vit('gc_vit_xxtiny', pretrained=pretrained, **model_kwargs)
    return model


@register_model
def gc_vit_xtiny(pretrained=False, **kwargs) -> GCViT:
    drop_path_rate = kwargs.pop("drop_path_rate", 0.2)
    model_kwargs = dict(depths=[3, 4, 6, 5],
                        num_heads=[2, 4, 8, 16],
                        window_size=[7, 7, 14, 7],
                        dim=64,
                        mlp_ratio=3,
                        drop_path_rate=drop_path_rate,
                        **kwargs
                        )
    model = _create_gc_vit('gc_vit_xtiny', pretrained=pretrained, **model_kwargs)
    return model


@register_model
def gc_vit_tiny(pretrained=False, **kwargs) -> GCViT:
    drop_path_rate = kwargs.pop("drop_path_rate", 0.2)
    model_kwargs = dict(depths=[3, 4, 19, 5],
                        num_heads=[2, 4, 8, 16],
                        window_size=[7, 7, 14, 7],
                        dim=64,
                        mlp_ratio=3,
                        drop_path_rate=drop_path_rate,
                        **kwargs
                        )
    model = _create_gc_vit('gc_vit_tiny', pretrained=pretrained, **model_kwargs)
    return model


@register_model
def gc_vit_tiny2(pretrained=False, **kwargs) -> GCViT:
    drop_path_rate = kwargs.pop("drop_path_rate", 0.25)
    model_kwargs = dict(depths=[3, 4, 29, 5],
                        num_heads=[2, 4, 8, 16],
                        window_size=[7, 7, 14, 7],
                        dim=64,
                        mlp_ratio=3,
                        drop_path_rate=drop_path_rate,
                        **kwargs
                        )
    model = _create_gc_vit('gc_vit_tiny2', pretrained=pretrained, **model_kwargs)
    return model


@register_model
def gc_vit_small(pretrained=False, **kwargs) -> GCViT:
    drop_path_rate = kwargs.pop("drop_path_rate", 0.3)
    model_kwargs = dict(depths=[3, 4, 19, 5],
                        num_heads=[3, 6, 12, 24],
                        window_size=[7, 7, 14, 7],
                        dim=96,
                        mlp_ratio=2,
                        drop_path_rate=drop_path_rate,
                        layer_scale=1e-5,
                        **kwargs
                        )
    model = _create_gc_vit('gc_vit_small', pretrained=pretrained, **model_kwargs)
    return model


@register_model
def gc_vit_small2(pretrained=False, **kwargs) -> GCViT:
    drop_path_rate = kwargs.pop("drop_path_rate", 0.35)
    model_kwargs = dict(depths=[3, 4, 23, 5],
                        num_heads=[3, 6, 12, 24],
                        window_size=[7, 7, 14, 7],
                        dim=96,
                        mlp_ratio=3,
                        drop_path_rate=drop_path_rate,
                        layer_scale=1e-5,
                        **kwargs
                        )
    model = _create_gc_vit('gc_vit_small2', pretrained=pretrained, **model_kwargs)
    return model


@register_model
def gc_vit_base(pretrained=False, **kwargs) -> GCViT:
    drop_path_rate = kwargs.pop("drop_path_rate", 0.5)
    model_kwargs = dict(depths=[3, 4, 19, 5],
                        num_heads=[4, 8, 16, 32],
                        window_size=[7, 7, 14, 7],
                        dim=128,
                        mlp_ratio=2,
                        drop_path_rate=drop_path_rate,
                        layer_scale=1e-5,
                        **kwargs
                        )
    
    
    model = _create_gc_vit('gc_vit_base', pretrained=pretrained, **model_kwargs)
#     in_features = model.head.in_features 
#     out_features = 5 # num_classes , set here
    
#     model.head = nn.Linear(in_features, out_features, bias=True)

    return model


@register_model
def gc_vit_large(pretrained=False, **kwargs) -> GCViT:
    drop_path_rate = kwargs.pop("drop_path_rate", 0.5)
    model_kwargs = dict(depths=[3, 4, 19, 5],
                        num_heads=[6, 12, 24, 48],
                        window_size=[7, 7, 14, 7],
                        dim=192,
                        mlp_ratio=2,
                        drop_path_rate=drop_path_rate,
                        layer_scale=1e-5,
                        **kwargs
                        )
    model = _create_gc_vit('gc_vit_large', pretrained=pretrained, **model_kwargs)
    return model


@register_model
def gc_vit_large_224_21k(pretrained=False, **kwargs) -> GCViT:
    drop_path_rate = kwargs.pop("drop_path_rate", 0.5)
    model_kwargs = dict(depths=[3, 4, 19, 5],
                        num_heads=[6, 12, 24, 48],
                        window_size=[7, 7, 14, 7],
                        dim=192,
                        mlp_ratio=2,
                        drop_path_rate=drop_path_rate,
                        layer_scale=1e-5,
                        **kwargs
                        )
    model = _create_gc_vit('gc_vit_large_224_21k', pretrained=pretrained, **model_kwargs)
    return model


@register_model
def gc_vit_large_384_21k(pretrained=False, **kwargs) -> GCViT:
    drop_path_rate = kwargs.pop("drop_path_rate", 0.1)
    model_kwargs = dict(depths=[3, 4, 19, 5],
                        num_heads=[6, 12, 24, 48],
                        window_size=[12, 12, 24, 12],
                        dim=192,
                        mlp_ratio=2,
                        drop_path_rate=drop_path_rate,
                        layer_scale=1e-5,
                        **kwargs
                        )
    model = _create_gc_vit('gc_vit_large_384_21k', pretrained=pretrained, **model_kwargs)
    return model


@register_model
def gc_vit_large_512_21k(pretrained=False, **kwargs) -> GCViT:
    drop_path_rate = kwargs.pop("drop_path_rate", 0.1)
    model_kwargs = dict(depths=[3, 4, 19, 5],
                        num_heads=[6, 12, 24, 48],
                        window_size=[16, 16, 32, 16],
                        dim=192,
                        mlp_ratio=2,
                        drop_path_rate=drop_path_rate,
                        layer_scale=1e-5,
                        **kwargs
                        )
    model = _create_gc_vit('gc_vit_large_512_21k', pretrained=pretrained, **model_kwargs)
    return model

In [15]:
import torch

class GCViT_Model(nn.Module): 
    def __init__(self, num_classes):
        super(GCViT_Model, self).__init__()
        # Add a convolutional layer at the top
        self.conv = nn.Conv2d(1, 3, kernel_size=3, stride=1, padding=1)  # Assuming input is grayscale (1 channel)

#         self.gcvit = gc_vit_large_512_21k(pretrained=False) # for image_size 512 x 512
        self.gcvit = gc_vit_base(pretrained=False) # for image_size 224 x 224


    def forward(self, x):
        x = self.conv(x)
        return self.gcvit(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")    
model = GCViT_Model(num_classes)
model.to(device)

# print(model)
print()

x = torch.randn(1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]).to(device)


output = model(x)
print("Model output's shape:", output.shape)
print(output) # logits 
display_params_flops(model)


/opt/conda/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /usr/local/src/pytorch/aten/src/ATen/native/TensorShape.cpp:3526.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



Model output's shape: torch.Size([1, 15])
tensor([[-0.1446, -0.0298, -0.1130, -0.2961, -0.3105,  0.1483,  0.3173, -0.1134,
         -0.0269, -0.2434, -0.1346,  0.1015,  0.0903, -0.4193, -0.3536]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Number of parameters in millions: 89.31 M
Number of trainable parameters in millions: 89.31 M
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_softmax() for <class 'torch.nn.modules.activation.Softmax'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
FL

### 6. EfficientViT

In [16]:
import timm
import torch.nn as nn

# efficientvit_b0.r224_in1k # is very fast 
# 'efficientvit_l3.r384_in1k # time consuming 

class EfficientViT(nn.Module):
    def __init__(self, num_classes, pretrained=True, fine_tune=False):
        super(EfficientViT, self).__init__()
        self.efficientvit = timm.create_model('efficientvit_b0.r224_in1k', 
           pretrained=pretrained,
           num_classes=num_classes,
           in_chans=1)
        
#         model.efficientvit.head
        if not fine_tune:
            for param in self.efficientvit.parameters():
                param.requires_grad = False
            
            for param in self.efficientvit.head.parameters():
                param.requires_grad = True
            


    def forward(self, x):
        return self.efficientvit(x)
    
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EfficientViT(num_classes, fine_tune=False)
model.to(device)

# print(model)
print()

x = torch.randn(1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]).to(device)



output = model(x)
print("Model output's shape:", output.shape)
print(output) # logits 
display_params_flops(model)

model.safetensors:   0%|          | 0.00/13.7M [00:00<?, ?B/s]


Model output's shape: torch.Size([1, 15])
tensor([[ 0.5227,  0.4816, -0.1879,  0.4157,  0.4021,  0.2044,  0.2763, -0.0605,
         -0.4593, -0.0542, -0.1839, -0.3059, -0.0949, -0.5918, -0.9585]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Number of parameters in millions: 2.15 M
Number of trainable parameters in millions: 1.47 M
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'

### 7. MaxViT

In [17]:
import timm
import torch.nn as nn

# maxvit_tiny_tf_224.in1k # for image size 224 x 224
# maxvit_tiny_tf_512.in1k # for image size 512 x 512



class MaxVit(nn.Module):
    def __init__(self, num_classes, pretrained=True, fine_tune=False):
        super(MaxVit, self).__init__()
        self.maxvit = timm.create_model('maxvit_tiny_tf_224.in1k', 
           pretrained=pretrained,
           num_classes=num_classes,
           in_chans=1)
        
        
        if not fine_tune:
            for param in self.maxvit.parameters():
                param.requires_grad = False
            
            for param in self.maxvit.head.parameters():
                param.requires_grad = True
            


    def forward(self, x):
        return self.maxvit(x)
    
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MaxVit(num_classes, fine_tune=False)
model.to(device)

# print(model)
print()

x = torch.randn(1, 1, IMAGE_SIZE[0], IMAGE_SIZE[1]).to(device)


output = model(x)
print("Model output's shape:", output.shape)
print(output) # logits 
display_params_flops(model)

model.safetensors:   0%|          | 0.00/124M [00:00<?, ?B/s]


Model output's shape: torch.Size([1, 15])
tensor([[-0.1084,  0.0446,  0.2134, -0.2865,  0.1619, -0.2049,  0.5878, -0.1520,
          0.0856,  0.0323,  0.0307, -0.1480, -0.2198,  0.0066,  0.0194]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Number of parameters in millions: 30.41 M
Number of trainable parameters in millions: 0.27 M
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
FLOPs: 5.33G, Params: 30.28M
